# parameter-subclass-of-tensor — worked example 3: freeze Parameters by name prefix

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `parameter-subclass-of-tensor`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Freezing a backbone flips `requires_grad=False` on every Parameter whose dotted name starts with a prefix, leaving the `Parameter` type tag intact so the params still show up in `parameters()`. The match is a literal `startswith`, not a regex.

## Worked solution

We reuse `Parameter` and provide a module with a `parameters()` generator yielding `(dotted_name, Parameter)` pairs. `freeze(module, prefix)` walks them and sets `p.requires_grad = False` for every name starting with `prefix`, mutating in place. We build a model with `encoder.fc1.weight`, `encoder.fc1.bias`, and `head.weight`, then freeze the `encoder.` prefix. Afterward the two encoder params are frozen while the head stays trainable, and all three are still Parameters. We print each name with its `requires_grad` to confirm the prefix-scoped freeze.

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class Parameter(MiniTensor):
    def __init__(self, array, requires_grad=True):
        super().__init__(array, requires_grad=requires_grad)

class Model:
    def __init__(self):
        self._params = [
            ('encoder.fc1.weight', Parameter([1.0])),
            ('encoder.fc1.bias', Parameter([0.0])),
            ('head.weight', Parameter([2.0])),
        ]
    def parameters(self):
        for name, p in self._params:
            yield name, p

def freeze(module, prefix):
    for name, p in module.parameters():
        if name.startswith(prefix):
            p.requires_grad = False

m = Model()
freeze(m, 'encoder.')
print([(name, p.requires_grad) for name, p in m.parameters()])